# Creador de Clips Virales - Acertijos para IG

---
## Conectar con Colab sin borrar el proyecto (usar GitHub)

1. **En tu PC:** sube cambios con `git add .`, `git commit -m "..."`, `git push` (repo: https://github.com/yoquelvisdev08/acertijo).
2. **En Colab:** Menu **File > Open notebook** > pestaña **GitHub** > pega `https://github.com/yoquelvisdev08/acertijo` o busca "acertijo".
3. Abre **creador_acertijos_ig.ipynb** desde la lista. Colab carga el notebook desde GitHub.
4. **Cuando cambies algo en tu PC:** haz `git add .`, `git commit -m "..."`, `git push`. Luego en Colab: **File > Open notebook > GitHub** y vuelve a abrir este notebook; tendras la version nueva. No hace falta borrar ni recrear nada.

**Opcional:** Si prefieres clonar el repo en Colab, ejecuta la **Celda 0** (abajo); luego abre este archivo desde `/content/acertijo/creador_acertijos_ig.ipynb`. Para traer cambios: ejecuta otra vez la Celda 0 (hace `git pull`) y reabre el notebook.

---

1. **Runtime > Change runtime type > GPU** (T4 o superior).
2. Ejecuta las celdas en orden.
3. TTS y video usan GPU por defecto. Si sale **CUDA device-side assert**, en Celda 2 pon `USE_CPU_FOR_TTS = True` para que solo la voz use CPU.
4. En la interfaz escribe el texto del acertijo y pulsa **Generar video**.
5. Descarga el MP4 desde el reproductor.

**Tiempo por video:** Primera vez 8-15 min (descarga voz + video). Siguientes ~6-11 min. No cierres mientras diga Processing.

**RAM:** Los modelos se cargan uno tras otro (voz -> se libera -> video) para no quedarse sin memoria. Revisa la **Consola (logs)** debajo de Estado para ver paso a paso que esta haciendo.

**Sobre "This share link expires in 1 week":** Es normal en Colab gratis. El enlace publico dura 7 dias. Para hosting permanente con GPU: `gradio deploy` en Hugging Face Spaces (https://huggingface.co/spaces).

In [ ]:
# Celda 0 (opcional): Clonar o actualizar desde GitHub para tener siempre la ultima version
# Pon la URL de tu repo y ejecuta esta celda. Luego abre este .ipynb desde la carpeta clonada.
REPO_URL = 'https://github.com/yoquelvisdev08/acertijo.git'
REPO_DIR = '/content/acertijo'

import os
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
    print('Repo actualizado. Si cambiaste el notebook, reabrelo desde File > Open notebook.')
else:
    !git clone {REPO_URL} {REPO_DIR}
    print(f'Clonado en {REPO_DIR}. Abre el notebook desde ahi: File > Open notebook > buscar creador_acertijos_ig.ipynb')

In [ ]:
# Celda 1: Instalacion (ejecutar una vez)
# requirements_colab.txt (integrado):
#   qwen-tts>=0.1.0, soundfile
#   diffusers>=0.34.0, transformers>=4.45.0, accelerate, ftfy
#   moviepy>=2.0.0, pysrt
#   gradio>=4.0.0
#   torch/torchvision ya en Colab
!apt-get update -qq && apt-get install -qq -y imagemagick sox > /dev/null
!pip install -q "qwen-tts>=0.1.0" soundfile "diffusers>=0.34.0" "transformers>=4.45.0" accelerate ftfy "moviepy>=2.0.0" pysrt "gradio>=4.0.0"
print('Dependencias instaladas.')

In [ ]:
# Celda 2: Imports y configuracion
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import torch
import numpy as np
import soundfile as sf
import re
from pathlib import Path

OUTPUT_DIR = Path('/content/salida_acertijos')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def sanitize_script(text: str, max_len: int = 500) -> str:
    if not text or not isinstance(text, str):
        return ''
    s = text.strip()
    s = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', s)
    return s[:max_len] if len(s) > max_len else s

def sanitize_video_prompt(text: str, max_len: int = 200) -> str:
    if not text or not isinstance(text, str):
        return 'Mysterious atmosphere, riddle, cinematic, vertical 9:16'
    s = text.strip()
    s = re.sub(r'[^a-zA-Z0-9\s,\.\-]', '', s)
    return (s[:max_len] if len(s) > max_len else s) or 'Mysterious atmosphere, riddle, cinematic, vertical 9:16'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_CPU_FOR_TTS = False
print(f'Dispositivo: {DEVICE}. TTS en GPU: {not USE_CPU_FOR_TTS} (pon USE_CPU_FOR_TTS=True en Celda 2 si da CUDA assert)')

In [ ]:
# Celda 3: Voz con Qwen3-TTS (espanol). Por defecto usa GPU; USE_CPU_FOR_TTS=True en Celda 2 para forzar CPU si hay CUDA assert.
def get_tts_model():
    import time
    print('[TTS] Estamos en: descargando/cargando modelo de VOZ (Qwen3-TTS). Este modelo narra el acertijo.', flush=True)
    t0 = time.time()
    from qwen_tts import Qwen3TTSModel
    tts_device = 'cpu' if USE_CPU_FOR_TTS else ('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[TTS] Dispositivo: {tts_device} (GPU = menos RAM, mas rapido). Descargando pesos si es la primera vez...', flush=True)
    model = Qwen3TTSModel.from_pretrained(
        'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice',
        device_map=tts_device,
        torch_dtype=torch.float16 if tts_device == 'cuda' else torch.float32,
    )
    print(f'[TTS] Modelo de voz listo en {time.time()-t0:.1f}s', flush=True)
    return model

SPEAKERS_VALID = ('Ryan', 'Vivian', 'Serena', 'Aiden', 'Ono_Anna', 'Sohee', 'Uncle_Fu', 'Dylan', 'Eric')

def generate_voice(text: str, out_path: str, speaker: str = 'Ryan', model=None):
    import time
    t0 = time.time()
    print('[TTS] Iniciando generacion de voz...', flush=True)
    if model is None:
        model = get_tts_model()
    text = sanitize_script(text, max_len=400)
    if not text:
        raise ValueError('Texto del acertijo vacio tras sanitizar.')
    speaker = speaker if speaker in SPEAKERS_VALID else 'Ryan'
    wavs, sr = model.generate_custom_voice(
        text=text,
        language='Spanish',
        speaker=speaker,
    )
    sf.write(out_path, wavs[0], sr)
    dur = len(wavs[0]) / sr
    print(f'[TTS] Listo en {time.time()-t0:.1f}s (audio {dur:.1f}s)', flush=True)
    return out_path, sr, dur

In [ ]:
# Celda 4: Video con Wan 2.1 T2V 1.3B (optimizado para Colab)
def get_video_pipeline():
    import time
    from diffusers import AutoModel, WanPipeline
    from diffusers.utils import export_to_video
    from transformers import UMT5EncoderModel
    model_id = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
    print('[VIDEO] Estamos en: descargando/cargando modelo de VIDEO (Wan 2.1). Este modelo genera las imagenes del video.', flush=True)
    print('[VIDEO] NO es el de voz. Primera vez puede tardar 2-5 min en descargar.', flush=True)
    t0 = time.time()
    print('[VIDEO] Paso 1/4: Descargando text_encoder...', flush=True)
    text_encoder = UMT5EncoderModel.from_pretrained(model_id, subfolder='text_encoder', torch_dtype=torch.bfloat16)
    print(f'[VIDEO] text_encoder listo ({time.time()-t0:.1f}s)', flush=True)
    print('[VIDEO] Paso 2/4: Descargando vae...', flush=True)
    vae = AutoModel.from_pretrained(model_id, subfolder='vae', torch_dtype=torch.float32)
    print(f'[VIDEO] vae listo ({time.time()-t0:.1f}s)', flush=True)
    print('[VIDEO] Paso 3/4: Descargando transformer...', flush=True)
    transformer = AutoModel.from_pretrained(model_id, subfolder='transformer', torch_dtype=torch.bfloat16)
    print(f'[VIDEO] transformer listo ({time.time()-t0:.1f}s)', flush=True)
    print('[VIDEO] Paso 4/4: Ensamblando pipeline y pasando a GPU...', flush=True)
    pipeline = WanPipeline.from_pretrained(
        model_id,
        vae=vae,
        transformer=transformer,
        text_encoder=text_encoder,
        torch_dtype=torch.bfloat16,
    )
    pipeline.to('cuda')
    print(f'[VIDEO] Pipeline cargado en GPU en {time.time()-t0:.1f}s', flush=True)
    return pipeline

def generate_video(prompt: str, out_path: str, num_frames: int = 49, pipeline=None):
    import time
    from diffusers.utils import export_to_video
    if pipeline is None:
        pipeline = get_video_pipeline()
    prompt = sanitize_video_prompt(prompt, max_len=150)
    num_frames = 49
    negative = 'static, blurry, low quality, text, subtitles, worst quality'
    print('[VIDEO] Generando frames (suele tardar 3-8 min, usa GPU)...', flush=True)
    t0 = time.time()
    out = pipeline(
        prompt=prompt,
        negative_prompt=negative,
        num_frames=num_frames,
        guidance_scale=5.0,
    ).frames[0]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print(f'[VIDEO] Frames generados en {time.time()-t0:.1f}s. Exportando MP4...', flush=True)
    export_to_video(out, out_path, fps=16)
    print(f'[VIDEO] Video guardado en {out_path}', flush=True)
    return out_path

In [ ]:
# Celda 5: Subtitulos y composicion (MoviePy)
def make_srt_from_script(script: str, duration_sec: float) -> str:
    lines = [s.strip() for s in script.replace('.', '. ').split('.') if s.strip()]
    if not lines:
        lines = [script]
    n = len(lines)
    step = duration_sec / n if n else duration_sec
    srt = []
    for i, line in enumerate(lines):
        start = i * step
        end = (i + 1) * step
        srt.append(f'{i+1}\n{_ts(start)} --> {_ts(end)}\n{line}\n')
    return '\n'.join(srt)

def _ts(sec: float) -> str:
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    ms = int((sec % 1) * 1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

def compose_video(video_path: str, audio_path: str, script: str, out_path: str, duration_audio: float):
    import time
    from moviepy.editor import VideoFileClip, AudioFileClip, CompositeVideoClip, TextClip
    from moviepy.config import change_settings
    print('[COMPOSE] Uniendo video + audio + subtitulos...', flush=True)
    t0 = time.time()
    try:
        change_settings({'IMAGEMAGICK_BINARY': '/usr/bin/convert'})
    except Exception:
        pass
    video = VideoFileClip(video_path)
    audio = AudioFileClip(audio_path)
    video = video.set_audio(audio)
    video = video.subclip(0, min(video.duration, audio.duration))
    target_h, target_w = 1080, 608
    if video.w > video.h:
        video = video.resize((target_w, target_h))
    print(f'[COMPOSE] Clips cargados ({time.time()-t0:.1f}s). Creando subtitulos...', flush=True)
    srt_text = make_srt_from_script(script, video.duration)
    lines = [x.strip() for x in script.split('.') if x.strip()] or [script]
    n = len(lines)
    step = video.duration / n if n else video.duration
    txt_clips = []
    for i, line in enumerate(lines):
        start = i * step
        end = (i + 1) * step
        try:
            tc = TextClip(line, fontsize=36, color='white', stroke_color='black', stroke_width=2)
            tc = tc.set_start(start).set_duration(step).set_position(('center', 'bottom'))
            txt_clips.append(tc)
        except Exception:
            pass
    if txt_clips:
        try:
            video = CompositeVideoClip([video] + txt_clips)
        except Exception:
            pass
    print('[COMPOSE] Escribiendo MP4 final...', flush=True)
    video.write_videofile(out_path, fps=24, codec='libx264', audio_codec='aac', verbose=False, logger=None)
    video.close()
    audio.close()
    print(f'[COMPOSE] Listo en {time.time()-t0:.1f}s', flush=True)
    return out_path

In [ ]:
# Celda 6: No precargamos modelos para ahorrar RAM. Se cargaran al generar (voz primero, luego video).
tts_model = None
print('Listo. Los modelos se descargaran/cargaran al pulsar Generar:', flush=True)
print('  1) Primero el modelo de VOZ (TTS) -> genera el audio.', flush=True)
print('  2) Se libera la voz para ahorrar RAM.', flush=True)
print('  3) Luego el modelo de VIDEO (Wan) -> genera las imagenes.', flush=True)
print('Revisa la consola de logs en la interfaz para ver cada paso.', flush=True)

In [ ]:
# Celda 7: Pipeline completo. run_pipeline_gen hace yield tras cada paso para que la consola se actualice en vivo.
video_pipeline = None

class TeeOut:
    def __init__(self, real, buf):
        self.real, self.buf = real, buf
    def write(self, s):
        self.real.write(s); self.buf.write(s)
    def flush(self):
        self.real.flush(); self.buf.flush()

def run_pipeline(texto_acertijo: str, prompt_video: str = None, speaker: str = 'Ryan'):
    for path, msg, _ in run_pipeline_gen(texto_acertijo, prompt_video, speaker):
        pass
    return path, msg

def run_pipeline_gen(texto_acertijo: str, prompt_video: str = None, speaker: str = 'Ryan'):
    global tts_model, video_pipeline
    import time
    import gc
    import sys
    from io import StringIO
    log_buf = StringIO()
    old_stdout, old_stderr = sys.stdout, sys.stderr
    sys.stdout = TeeOut(old_stdout, log_buf)
    sys.stderr = TeeOut(old_stderr, log_buf)
    t0 = time.time()
    try:
        print('='*60, flush=True)
        print('[PIPELINE] INICIO. Consola activa: se actualiza en cada paso.', flush=True)
        print('='*60, flush=True)
        yield None, 'Iniciando pipeline...', log_buf.getvalue()
        if not texto_acertijo or not texto_acertijo.strip():
            yield None, 'Escribe el texto del acertijo.', log_buf.getvalue()
            return
        script = sanitize_script(texto_acertijo)
        if not script:
            yield None, 'Texto no valido.', log_buf.getvalue()
            return
        prompt_video = sanitize_video_prompt(prompt_video) if prompt_video else 'Mysterious atmosphere, riddle, cinematic, vertical 9:16'
        base = OUTPUT_DIR / f'clip_{int(time.time())}'
        base.parent.mkdir(parents=True, exist_ok=True)
        audio_path = str(base) + '_audio.wav'
        video_path = str(base) + '_video.mp4'
        final_path = str(base) + '_final.mp4'
        duration_audio = 0.0
        try:
            yield None, 'Paso 1/3 - VOZ (cargando/generando audio)...', log_buf.getvalue()
            if tts_model is None:
                print('[PIPELINE] Paso 1/3 - VOZ: Cargando modelo de VOZ (TTS)...', flush=True)
                tts_model = get_tts_model()
            else:
                print('[PIPELINE] Paso 1/3 - VOZ: Generando audio...', flush=True)
            _, _, duration_audio = generate_voice(script, audio_path, speaker=speaker, model=tts_model)
            print(f'[PIPELINE] Paso 1/3 listo. Tiempo: {time.time()-t0:.1f}s', flush=True)
            print('[PIPELINE] Liberando modelo de voz para ahorrar RAM...', flush=True)
            tts_model = None
            gc.collect()
            yield None, 'Paso 1/3 listo. Liberando voz.', log_buf.getvalue()
        except Exception as e:
            yield None, f'Error en voz: {e}', log_buf.getvalue()
            return
        try:
            yield None, 'Paso 2/3 - VIDEO (cargando/generando imagenes)...', log_buf.getvalue()
            if video_pipeline is None:
                print('[PIPELINE] Paso 2/3 - VIDEO: Cargando modelo de VIDEO (Wan)...', flush=True)
                video_pipeline = get_video_pipeline()
            else:
                print('[PIPELINE] Paso 2/3 - VIDEO: Generando imagenes...', flush=True)
            generate_video(prompt_video, video_path, num_frames=49, pipeline=video_pipeline)
            print(f'[PIPELINE] Paso 2/3 listo. Tiempo: {time.time()-t0:.1f}s', flush=True)
            yield None, 'Paso 2/3 listo.', log_buf.getvalue()
        except Exception as e:
            yield None, f'Error en video: {e}', log_buf.getvalue()
            return
        try:
            yield None, 'Paso 3/3 - Composicion final...', log_buf.getvalue()
            print('[PIPELINE] Paso 3/3 - COMPOSICION...', flush=True)
            compose_video(video_path, audio_path, script, final_path, duration_audio)
            print(f'[PIPELINE] FIN. Tiempo total: {time.time()-t0:.1f}s', flush=True)
            yield final_path, f'Listo en {time.time()-t0:.1f}s', log_buf.getvalue()
        except Exception as e:
            yield None, f'Error al componer: {e}', log_buf.getvalue()
    except Exception as e:
        import traceback
        traceback.print_exc()
        yield None, str(e), log_buf.getvalue()
    finally:
        sys.stdout, sys.stderr = old_stdout, old_stderr


In [ ]:
# Celda 8: Interfaz Gradio. generar es un generador que hace yield en cada paso para que la Consola se actualice en vivo.
import gradio as gr

def generar(texto, prompt_video, speaker):
    for path, msg, logs in run_pipeline_gen(texto, prompt_video, speaker):
        yield path, msg, logs

demo = gr.Interface(
    fn=generar,
    inputs=[
        gr.Textbox(label='Texto del acertijo', placeholder='Ej: Que cosa tiene dientes y no muerde? La sierra.'),
        gr.Textbox(label='Descripcion del video (opcional)', value='Mysterious atmosphere, riddle, enigma, dark mood, cinematic, vertical', lines=2),
        gr.Dropdown(choices=['Ryan', 'Vivian', 'Serena', 'Aiden', 'Ono_Anna', 'Sohee'], value='Ryan', label='Voz'),
    ],
    outputs=[
        gr.Video(label='Video'),
        gr.Textbox(label='Estado'),
        gr.Textbox(label='Consola (logs)', lines=18, max_lines=30),
    ],
    title='Creador de Clips - Acertijos IG',
    description='Genera un video con voz y subtitulos. Suele tardar 6-11 min. Revisa la consola debajo si hay error.',
)

In [ ]:
# Celda 9: Lanzar interfaz (solo link publico, no embebido en Colab)
demo.launch(share=True, inline=False)